In [ ]:
"""MedScope_AI_Colab_FullSystem (2).ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1_oK9_nx31hi6K8Nt1lxII3iGIjx3-8Yv

# MedScope AI — Full System Inference on Colab

This notebook pulls code from GitHub, loads your trained weights from Google Drive, and starts both the Backend API and the Next.js Frontend using localtunnel.
"""

In [ ]:
#@title 1. Runtime and configuration
import os, sys
REPO_URL = "https://github.com/prey801/finalyearproject.git"
BRANCH = "main"
PROJECT_DIR = "/content/finalyearproject"
DRIVE_WEIGHTS_DIR = "/content/drive/MyDrive/medscope-ai/weights"

In [ ]:
SUPABASE_URL = "YOUR_SUPABASE_URL" #@param {type:"string"}
SUPABASE_ANON_KEY = "YOUR_SUPABASE_ANON_KEY" #@param {type:"string"}
GITHUB_TOKEN = "YOUR_GITHUB_TOKEN" #@param {type:"string"}


In [ ]:
#@title 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
#@title 3. Clone Repository
import os
%cd /content
!rm -rf "$PROJECT_DIR"
!git clone --branch "$BRANCH" "$REPO_URL" "$PROJECT_DIR"

In [ ]:
if os.path.exists(PROJECT_DIR):
    print(f"\n✅ Repository cloned to {PROJECT_DIR}")
#     %cd "$PROJECT_DIR"
    !ls -F
else:
    print("\n❌ Failed to clone repository.")

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
#@title 4. Install localtunnel & SAM2
!npm install -g localtunnel
%cd $PROJECT_DIR
!wget -q -O sam2_hiera_small.pt https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
print("SAM2 weights downloaded!")

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
#@title 5. Install Backend & Frontend Dependencies
!apt-get update -qq
!apt-get install -y -qq redis-server postgresql libgl1

In [ ]:
# Upgrade core python tools and force critical versions to stop ModuleNotFoundErrors
!python -m pip install -q -U pip setuptools wheel
print("Installing backend and model dependencies...")
# Adjusted versions to satisfy common Colab conflicts while meeting project requirements
!python -m pip install -q --force-reinstall "fastapi>=0.115.2" "pydantic>=2.7.0" "starlette>=0.36.3" "uvicorn>=0.34.0" "python-multipart>=0.0.18"
!python -m pip install -q -r backend/requirements.txt
!python -m pip install -q -r models/requirements.txt
!python -m pip install -q pytest qdrant-client nest_asyncio pyngrok

In [ ]:
# Install Frontend dependencies
print("Installing frontend dependencies...")
%cd $PROJECT_DIR/frontend
!rm -rf node_modules .next
!npm install --legacy-peer-deps
%cd $PROJECT_DIR
print("\n✅ All dependencies installed successfully.")

In [ ]:
#@title 6. Load Trained Weights from Drive
import shutil
from pathlib import Path
src_dir = Path(DRIVE_WEIGHTS_DIR)
dst_dir = Path(PROJECT_DIR) / "models/weights"
dst_dir.mkdir(parents=True, exist_ok=True)
for p in src_dir.glob("*.pt"):
    print(f"Copying {p.name}...")
    shutil.copy2(p, dst_dir / p.name)
for p in src_dir.glob("*.pth"):
    print(f"Copying {p.name}...")
    shutil.copy2(p, dst_dir / p.name)
print("Weights loaded!")

In [ ]:
import os
# @title 8b. Configure ngrok Authtoken
NGROK_AUTHTOKEN = "YOUR_NGROK_AUTHTOKEN" #@param {type:"string"}


In [ ]:
if NGROK_AUTHTOKEN:
    !ngrok config add-authtoken {NGROK_AUTHTOKEN}
    print("✅ ngrok authtoken configured!")
else:
    print("⚠️ Please enter your ngrok authtoken to proceed.")

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
#@title 7 & 8. Start Infrastructure and Run the Entire System
import os, subprocess, time, urllib.request
from pathlib import Path

In [ ]:
# --- 1. Infrastructure (Redis, Postgres, Qdrant) ---
print("Starting Infrastructure Services...")
!service redis-server start
!service postgresql start

In [ ]:
# Set up PostgreSQL
!sudo -u postgres psql -c "CREATE USER medscope WITH PASSWORD 'password';" || true
!sudo -u postgres psql -c "CREATE DATABASE medscope_db OWNER medscope;" || true

In [ ]:
# Set up Qdrant
%cd $PROJECT_DIR
if not Path("qdrant").exists():
    !wget -q https://github.com/qdrant/qdrant/releases/download/v1.7.0/qdrant-x86_64-unknown-linux-gnu.tar.gz
    !tar -xzf qdrant-x86_64-unknown-linux-gnu.tar.gz

In [ ]:
if not subprocess.run(["pgrep", "-f", "qdrant"], capture_output=True).stdout:
    subprocess.Popen(["./qdrant"])
time.sleep(3)
print("✅ Infrastructure services started.")

In [ ]:
# --- 2. Application Setup ---
import pyngrok
from pyngrok import ngrok

In [ ]:
# Set environment variables for build and runtime
os.environ['YOLO_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/yolov11_malaria_best.pt")
os.environ['SWIN_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "models/weights/swin_malaria_best.pth")
os.environ['SAM2_WEIGHTS_PATH'] = str(Path(PROJECT_DIR) / "sam2_hiera_small.pt")
os.environ['ALLOWED_ORIGINS'] = "*"

In [ ]:
# CRITICAL: Force keys into system environment for Next.js build prerendering
if "SUPABASE_URL" in globals():
    os.environ['NEXT_PUBLIC_SUPABASE_URL'] = SUPABASE_URL
if "SUPABASE_ANON_KEY" in globals():
    os.environ['NEXT_PUBLIC_SUPABASE_ANON_KEY'] = SUPABASE_ANON_KEY

In [ ]:
if "NGROK_AUTHTOKEN" in globals() and NGROK_AUTHTOKEN:
    ngrok.set_auth_token(NGROK_AUTHTOKEN)

In [ ]:
# Cleanup old processes
for proc in ['uvicorn', 'celery', 'ngrok', 'npx', 'next']:
    !pkill -9 -f {proc} || true
time.sleep(2)

In [ ]:
# --- STEP 1: Start Backend ---
print("\n[1/5] Starting FastAPI backend...")
with open("backend_startup.log", "w") as f_log:
    backend_process = subprocess.Popen(
        ["uvicorn", "backend.main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd=PROJECT_DIR, stdout=f_log, stderr=f_log
    )

In [ ]:
print("   Waiting for backend to be ready...")
for i in range(12):
    try:
        urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2)
        print(f"   ✅ Backend ready!")
        break
    except Exception:
        time.sleep(5)
        if i % 2 == 0: print(f"   ⏳ Still waiting... ({i*5}s)")
else:
    print("   ⚠️ Backend timed out. Checking backend_startup.log:")
    !tail -n 20 backend_startup.log

In [ ]:
# --- STEP 2: Start Celery Worker ---
print("[2/5] Starting Celery worker...")
with open("celery_startup.log", "w") as f_log:
    subprocess.Popen(["celery", "-A", "backend.worker", "worker", "--loglevel=info"], cwd=PROJECT_DIR, stdout=f_log, stderr=f_log)

In [ ]:
# --- STEP 3: ngrok Tunnel ---
print("[3/5] Opening ngrok tunnel for backend...")
backend_tunnel = ngrok.connect(8000, bind_tls=True)
backend_url = backend_tunnel.public_url
print(f"   ⚙️  Backend: {backend_url}")

In [ ]:
# --- STEP 4: Write Config ---
with open(f"{PROJECT_DIR}/frontend/.env.local", "w") as f:
    f.write(f"NEXT_PUBLIC_API_URL={backend_url}\n")
    f.write(f"NEXT_PUBLIC_SUPABASE_URL={os.environ.get('NEXT_PUBLIC_SUPABASE_URL','')}\n")
    f.write(f"NEXT_PUBLIC_SUPABASE_ANON_KEY={os.environ.get('NEXT_PUBLIC_SUPABASE_ANON_KEY','')}\n")

In [ ]:
# --- STEP 5: Build and Start Frontend ---
print("[5/5] Building & Starting Next.js frontend...")
try:
    !rm -rf {PROJECT_DIR}/frontend/.next
    # Explicitly pass os.environ to ensure Next.js build sees the Supabase keys
    build_res = subprocess.run(["npm", "run", "build"], cwd=f"{PROJECT_DIR}/frontend", capture_output=True, text=True, env=os.environ)
    if build_res.returncode != 0:
        print("❌ Frontend build failed. Error logs:")
        print(build_res.stderr)
    else:
        print("   ✅ Frontend built successfully!")
        subprocess.Popen(["npx", "next", "start", "-H", "0.0.0.0", "-p", "3000"], cwd=f"{PROJECT_DIR}/frontend")
        time.sleep(10)
        frontend_tunnel = ngrok.connect(3000, bind_tls=True, host_header="rewrite")
        print(f"\n🚀 SYSTEM IS RUNNING!\n🌍 Frontend: {frontend_tunnel.public_url}\n⚙️  Backend:  {backend_url}")
except Exception as e:
    print(f"❌ Startup error: {e}")

In [ ]:
# Commented out IPython magic to ensure Python compatibility.
import os

In [ ]:
print(f"Pulling latest changes in {PROJECT_DIR}...")
%cd {PROJECT_DIR}
!git pull
print("Git pull complete. Please re-run the 'Run the Entire System' cell to apply changes.")

In [ ]:
import os
print('--- Current Working Directory ---')
!pwd

In [ ]:
print('\n--- File Structure (Top Level) ---')
!ls -F /content/finalyearproject

In [ ]:
print('\n--- Checking for critical files ---')
for path in ['backend/requirements.txt', 'frontend/package.json', 'models/requirements.txt']:
    full_path = os.path.join('/content/finalyearproject', path)
    exists = os.path.exists(full_path)
    print(f"{path}: {'✅ Found' if exists else '❌ Missing'}")

In [ ]:
print('\n--- Backend Startup Log (if exists) ---')
if os.path.exists('/content/finalyearproject/backend_startup.log'):
    !tail -n 20 /content/finalyearproject/backend_startup.log
else:
    print('backend_startup.log not found.')

In [ ]:
"""### Cleanup Running Processes

This cell will attempt to stop any currently running backend, Celery, ngrok, or frontend processes. Note that zombie processes (marked `<defunct>`) cannot be directly killed; a runtime restart is usually required to clear them if they persist.
"""

In [ ]:
import time

In [ ]:
print("Terminating active backend, Celery, ngrok, and frontend processes...")
for proc_name in ['uvicorn', 'celery', 'ngrok', 'npm']: # Added 'npm' for frontend cleanup
    !pkill -f {proc_name} || true
time.sleep(2) # Give processes a moment to terminate
print("Cleanup complete.")
print("If you still see '<defunct>' processes in `ps aux`, consider restarting the Colab runtime.")